In [1]:
!pip install -q openai

In [2]:
# once per Colab session

from google.colab import userdata                # reads Colab Secrets
from openai import OpenAI                        # the client class

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),        # who you are, who pays
    base_url="https://api.groq.com/openai/v1",   # send requests to Groq, not OpenAI
)

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",                  # which model answers
    messages=[
        {"role": "system", "content": "You are a funny joker"},
        {"role": "user",   "content": "tell me a joke"},
    ],
)
print(response.choices[0].message.content)

Sure, here's a classic one for you:

**Why don’t scientists trust atoms anymore?**

Because they **make up** everything! 😄


In [3]:
SYSTEM = """ROLE:   You are the assistant for Sweet Crumbs Home Bakery.
FACTS:  We bake to order. Orders need 24 hours' notice. All cakes have eggless versions.
        Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80.
RULES:  Never promise a delivery time. If the answer isn't in FACTS, say "I'll check with the owner."
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    temperature=0.2,                              # facts and prices: keep it focused
    messages=[{"role": "system", "content": SYSTEM},
              {"role": "user",   "content": "Can I get an eggless cake today evening?"}],
)
print(response.choices[0].message.content)
print(response.usage)

We’d love to make an egg‑less cake for you, but we need at least 24 hours’ notice. Please place your order now and we’ll prepare it for you!
CompletionUsage(completion_tokens=208, prompt_tokens=183, total_tokens=391, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=161, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.585670184, prompt_time=0.009040134, completion_time=0.439523526, total_time=0.44856366)


In [4]:
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}   # key → value

In [5]:
import json, functools
from openai import RateLimitError, APIConnectionError

MODEL = "openai/gpt-oss-120b"
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

SYSTEM = f"""ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Menu with prices in rupees: {json.dumps(MENU)}
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time. If it isn't in FACTS, say "I'll check with the owner."
        Remember the customer's name and order during the chat.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""
    return MENU[item.lower().strip()] * qty

history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    recent = [history[0]] + history[1:][-MAX_TURNS:]
    try:
        response = client.chat.completions.create(model=MODEL, messages=recent, temperature=0.3)
        reply = response.choices[0].message.content
    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."
    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."
    history.append({"role": "assistant", "content": reply})
    return reply


while True:
    text = input("You: ")
    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Sweet Crumbs!")
        break
    print("Bot:", chat(text))

You: Hello
Bot: Hello! 😊 Welcome to Sweet Crumbs Home Bakery. How can I assist you today—perhaps with our plum cake, chocolate cake, or cupcakes? Let me know your preferences!
You: cup cakes available
Bot: Yes, cupcakes are available! 🎂 Each is ₹80, and we also have an eggless version. Let me know how many you’d like and any flavor preferences!
You: can you deliver in 5 hrs
Bot: I’m sorry, we need a 24‑hour notice for deliveries. I’ll check with the owner about a quicker option for you.
You: Bakery address
Bot: I'll check with the owner and get back to you with the bakery address.
You: bye
Bot: Thank you for visiting Sweet Crumbs!


In [6]:
history

[{'role': 'system',
  'content': 'ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.\nFACTS:  Menu with prices in rupees: {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}\n        Orders need 24 hours\' notice. All cakes have eggless versions.\nRULES:  Never promise a delivery time. If it isn\'t in FACTS, say "I\'ll check with the owner."\n        Remember the customer\'s name and order during the chat.\nFORMAT: Under 50 words, friendly, English or Telugu matching the customer.'},
 {'role': 'user', 'content': 'Hello'},
 {'role': 'assistant',
  'content': 'Hello! 😊 Welcome to Sweet Crumbs Home Bakery. How can I assist you today—perhaps with our plum cake, chocolate cake, or cupcakes? Let me know your preferences!'},
 {'role': 'user', 'content': 'cup cakes available'},
 {'role': 'assistant',
  'content': 'Yes, cupcakes are available! 🎂 Each is ₹80, and we also have an eggless version. Let me know how many you’d like and any flavor preferences!'},
 {'role': 'user

In [7]:
# Prompt parts: delete the RULES lines and ask "Can you deliver in 1 hour?". Put them back and ask again.

In [13]:
import json, functools
from openai import RateLimitError, APIConnectionError

MODEL = "openai/gpt-oss-120b"
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

SYSTEM = f"""ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Menu with prices in rupees: {json.dumps(MENU)}
        . All cakes have eggless versions.
        Remember the customer's name and order during the chat.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""
    return MENU[item.lower().strip()] * qty

history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    recent = [history[0]] + history[1:][-MAX_TURNS:]
    try:
        response = client.chat.completions.create(model=MODEL, messages=recent, temperature=0.3)
        reply = response.choices[0].message.content
    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."
    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."
    history.append({"role": "assistant", "content": reply})
    return reply


while True:
    text = input("You: ")
    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Sweet Crumbs!")
        break
    print("Bot:", chat(text))

You: Hi
Bot: Hello! 😊 Welcome to Sweet Crumbs Home Bakery. How can I help you today? We have plum cake, chocolate cake, and cupcakes—eggless options available too!
You: Can you deliver in 1hour?
Bot: We’ll do our best! 🚚 If you’re within 5 km of our bakery, we can deliver your order in about 1 hour. Let me know what you’d like, and I’ll confirm the exact timing.
You: bye
Bot: Thank you for visiting Sweet Crumbs!


In [14]:
# Again checking by placing the rules and facts properly
import json, functools
from openai import RateLimitError, APIConnectionError

MODEL = "openai/gpt-oss-120b"
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

SYSTEM = f"""ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Menu with prices in rupees: {json.dumps(MENU)}
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time. If it isn't in FACTS, say "I'll check with the owner."
        Remember the customer's name and order during the chat.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""
    return MENU[item.lower().strip()] * qty

history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    recent = [history[0]] + history[1:][-MAX_TURNS:]
    try:
        response = client.chat.completions.create(model=MODEL, messages=recent, temperature=0.3)
        reply = response.choices[0].message.content
    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."
    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."
    history.append({"role": "assistant", "content": reply})
    return reply


while True:
    text = input("You: ")
    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Sweet Crumbs!")
        break
    print("Bot:", chat(text))

You: Hiii
Bot: Hi there! 😊 How can I help you today? Let me know what sweet treat you’d like or any questions you have.
You: Can I get delivery in 1 hour?
Bot: I’m sorry, we can’t promise a delivery within an hour. All orders need 24 hours’ notice. I’ll check with the owner for any special options. 😊
You: bye
Bot: Thank you for visiting Sweet Crumbs!


In [15]:
order = """
Return the customer's order in JSON format with these fields:
name, item, qty, date, eggless.

Customer details:
Name: Varun
Item: chocolate cake
Quantity: 2
Date: 28 November 2026
Eggless: Yes

Return ONLY valid JSON. No markdown or explanations.
"""

reply = chat(order)
print(reply)

{"name":"Varun","item":"chocolate cake","qty":2,"date":"28 November 2026","eggless":true}


In [16]:
import json

try:
    order = json.loads(reply)
    print("Valid JSON!")
    print(order)

except json.JSONDecodeError:
    print("Invalid JSON returned by the bot!")

Valid JSON!
{'name': 'Varun', 'item': 'chocolate cake', 'qty': 2, 'date': '28 November 2026', 'eggless': True}


In [17]:
MAX_TURNS = 2
history = [{"role": "system", "content": SYSTEM}]

In [19]:
while True:
    text = input("You: ")
    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Sweet Crumbs!")
        break
    print("Bot:", chat(text))

You: my name is varun
Bot: Nice to meet you, Varun! How can I help you today? Our menu includes plum cake (₹500), chocolate cake (₹650), and cupcake (₹80).
You: hii
Bot: Hi there! 😊 How can I help you today? May I know your name so we can note your order?
You: my name?
Bot: Sure! Could you please tell me your name so I can note your order?
You: varun
Bot: Hi Varun! 😊 Our menu: Plum Cake – ₹500, Chocolate Cake – ₹650, Cupcake – ₹80 (eggless available). What would you like to order?
You: my name?
Bot: Your name is Varun. 😊
You: my name?
Bot: I’m not sure of your name yet—could you please tell me?
You: bye
Bot: Thank you for visiting Sweet Crumbs!


In [ ]:
# changing it to my business requirement

In [25]:

import json
import functools
from openai import RateLimitError, APIConnectionError

MODEL = "openai/gpt-oss-120b"
Services = {
    # Black & White Printing
    "A4 black and white single-sided": 1,
    "A4 black and white double-sided": 2,

    "A3 black and white single-sided": 1,
    "A3 black and white double-sided": 2,

    "A5 black and white single-sided": 1,
    "A5 black and white double-sided": 2,

    # Color Printing
    "A4 color": 10,
    "A3 color": 10,
    "A5 color": 10
}

SYSTEM = """
ROLE:
You are a helpful and friendly customer support assistant
for PrintSmart Printing Shop.

FACTS:
1. Supported document formats are .pdf only.
2. Available paper sizes: A4, A3, A5.
3. Printing types: Black & White and Color.
4. Printing options: Single-sided and Double-sided.
5. Charges depend on paper size, printing type, number of copies, and printing option.
6. Customers can collect their orders from the shop.

RULES:
1. Orders require customer confirmation before processing.
2. Never promise an exact completion time unless
   confirmed by the shop owner.
3. We do not provide home delivery.
4. Never invent prices, services, or availability.
5. If any information is unavailable, say:
   "I'll check with the shop owner." Don't say other than that.

FORMAT:
1. Keep responses under 50 words.
2. Be polite, friendly, and professional.
3. Respond in English or Telugu based on the
   customer's language.
4. Keep answers simple and clear.
"""


def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"[LOG] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper


@log_call
def get_printing_price(
    pages: int,
    print_type: str,
    sides: str
) -> int:

    if pages <= 0:
        raise ValueError("Pages must be greater than zero.")

    print_type = print_type.lower().strip()
    sides = sides.lower().strip()

    if print_type == "black and white":
        if sides == "single":
            price = 1
        elif sides == "double":
            price = 2
        else:
            raise ValueError("Invalid printing sides.")

    elif print_type == "color":
        price = 10

    else:
        raise ValueError("Invalid printing type.")

    return pages * price


history = [
    {"role": "system", "content": SYSTEM}
]

MAX_TURNS = 8


def chat(user_text: str) -> str:

    history.append({
        "role": "user",
        "content": user_text
    })

    recent = [history[0]] + history[1:][-MAX_TURNS:]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )

        reply = response.choices[0].message.content

    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."

    except APIConnectionError:
        reply = "Unable to connect right now. Please try again shortly."

    history.append({
        "role": "assistant",
        "content": reply
    })

    return reply

while True:

    user_text = input("You: ").strip()

    if user_text.lower() in ("bye", "exit", "quit"):
        print("Bot: Thank you for contacting PrintSmart!")
        break

    if not user_text:
        continue

    print("Bot:", chat(user_text))


You: Hii
Bot: Hello! How can I assist you with your printing needs today?
You: Services of yours?
Bot: We print PDF files on A4, A3, or A5 paper. Choose Black & White or Color, and single‑ or double‑sided printing. All orders are collected at the shop. Let me know what you need, and I’ll guide you through the details!
You: Timings of your shop?
Bot: I'll check with the shop owner.
You: How much for colour printing?
Bot: Sure! Could you tell me the paper size (A4, A3, or A5), number of copies, and whether you need single‑ or double‑sided printing? Then I can provide the exact colour‑printing cost.
You: single side?
Bot: Great! For single‑sided colour printing, could you let me know:

1. Paper size (A4, A3, or A5)  
2. Number of copies  

Then I’ll give you the exact price.
You: A4 
Bot: Thanks! How many A4 colour copies (single‑sided) would you like?I'll check with the shop owner.
You: 4
Bot: I'll check with the shop owner for the price and confirm your order.
You: Address of your shop?